In [18]:
def predict_college(student_rank, selected_round, student_category, df):
    """
    Predicts colleges based on an adjusted student rank (original rank increased by 10%)
    and selected round, suggesting all institutes with cutoff ranks better than or equal to this adjusted rank.
    Also filters by student category and displays the yearly fee.
    """
    # Calculate the adjusted rank (original rank increased by 10%)
    adjusted_student_rank = student_rank * 1.10

    # Determine the rank and score columns based on the selected round
    rank_col = f'{selected_round} Rank'
    score_col = f'{selected_round} Score'

    # Filter colleges where their cutoff rank is better than or equal to the adjusted student rank
    # (i.e., college_rank >= adjusted_student_rank)
    base_eligible_colleges = df[
        (df[rank_col] >= adjusted_student_rank) &
        (df[rank_col].notna())
    ].copy() # Use .copy() to avoid SettingWithCopyWarning

    # Apply category filter logic
    if student_category == 'All':
        eligible_colleges = base_eligible_colleges.copy()
    else:
        # Include colleges from the specified student category
        specific_category_colleges = base_eligible_colleges[base_eligible_colleges['Category'] == student_category].copy()
        # Include colleges from the 'OPEN' category as well
        open_category_colleges = base_eligible_colleges[base_eligible_colleges['Category'] == 'OPEN'].copy()
        # Combine and remove duplicates (if any college appears in both)
        eligible_colleges = pd.concat([specific_category_colleges, open_category_colleges]).drop_duplicates().copy()

    if not eligible_colleges.empty:
        # Add a 'Chance' column based on the college's rank relative to student's original and adjusted rank
        def get_chance(college_rank, student_rank_original, adjusted_rank):
            if college_rank < student_rank_original: # College cutoff is better than original student rank
                return "High Chance"
            elif college_rank < student_rank_original * 1.05: # College cutoff is within 5% of original student rank
                return "Moderate Chance"
            else: # College cutoff is between 5% and 10% of original student rank (up to adjusted rank)
                return "Low Chance"

        eligible_colleges['Chance'] = eligible_colleges[rank_col].apply(
            lambda x: get_chance(x, student_rank, adjusted_student_rank)
        )

        print(f"\nColleges eligible for student rank {student_rank} (adjusted to {adjusted_student_rank:.0f}) in {selected_round} for category {student_category} (including OPEN category institutes):")
        # Sort by rank_col and display relevant columns, excluding 'Chance'
        display(eligible_colleges.sort_values(by=rank_col)[['Institute Name', 'State', 'Category', rank_col, 'Fee']])
    else:
        print(f"\nNo colleges found with cutoff rank better than or equal to {adjusted_student_rank:.0f} in {selected_round} for category {student_category} (including OPEN category institutes).")

print("College prediction function updated.")

College prediction function updated.


In [19]:
import pandas as pd

# Load the Excel file into a DataFrame, specifying the header row where actual column names are located
file_path = '/content/Aman_TAB_India_Deemed_MBBS_Cutoff_2025.xlsx'
df = pd.read_excel(file_path, header=2) # header=2 means the third row (0-indexed) has the correct headers

# Drop the first row which contains 'SN' and empty values as it's not useful as a header
df = df.iloc[1:].copy() # Keep all rows from index 1 onwards

# Rename columns for clarity and consistency, using the correct 'Unnamed' column names from the loaded DataFrame
df.rename(columns={'Unnamed: 0': 'SN',
                   'Unnamed: 1': 'Institute Name',
                   'Unnamed: 2': 'State',
                   'Unnamed: 3': 'Fee',
                   'Unnamed: 4': 'Category',
                   'Unnamed: 5': 'R1 Rank',
                   'Unnamed: 6': 'R1 Score',
                   'Unnamed: 7': 'R2 Rank',
                   'Unnamed: 8': 'R2 Score',
                   'Unnamed: 9': 'R3 Rank',
                   'Unnamed: 10': 'R3 Score'},
          inplace=True)

# Convert 'XX' values to NaN in rank and score columns and convert to numeric
rank_score_cols = ['R1 Rank', 'R1 Score', 'R2 Rank', 'R2 Score', 'R3 Rank', 'R3 Score', 'Fee']
for col in rank_score_cols:
    df[col] = df[col].replace('XX', pd.NA)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Display the first few rows and column information to verify loading and inspect columns
print("DataFrame loaded and cleaned successfully. First 5 rows:")
display(df.head())
print("\nDataFrame Info:")
df.info()

DataFrame loaded and cleaned successfully. First 5 rows:


,SN,Institute Name,State,Fee,Category,R1 Rank,R1 Score,R2 Rank,R2 Score,R3 Rank,R3 Score
1,1,GITAM Visakhapatnam,Andhra Pradesh,2537000.0,OPEN,437866.0,302.0,482169.0,287.0,615974.0,246.0
2,2,Hamdard HIMSR (MM QUOTA),Delhi,1600000.0,Muslim Minority,NaN,NaN,NaN,NaN,NaN,NaN
3,2,"Hamdard HIMSR, New Delhi",Delhi,1600000.0,OPEN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,SBKS Vidyapeeth,Gujarat,2275000.0,OPEN,340315.0,340.0,398037.0,317.0,535000.0,270.0
5,3,SBKS Vidyapeeth JM (JAIN QUOTA),Gujarat,2275000.0,Jain Minority,681238.0,229.0,785540.0,203.0,1087718.0,145.0



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 1 to 62
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   SN              62 non-null     object 
 1   Institute Name  62 non-null     object 
 2   State           62 non-null     object 
 3   Fee             61 non-null     float64
 4   Category        62 non-null     object 
 5   R1 Rank         57 non-null     float64
 6   R1 Score        57 non-null     float64
 7   R2 Rank         59 non-null     float64
 8   R2 Score        59 non-null     float64
 9   R3 Rank         60 non-null     float64
 10  R3 Score        60 non-null     float64
dtypes: float64(7), object(4)
memory usage: 5.5+ KB


In [22]:
#@title Predict Colleges
student_rank_input = 300000 #@param {type:"number"}
selected_round_input = 'R2' #@param ['R1', 'R2', 'R3']
student_category_input = 'Jain Minority' #@param ['OPEN', 'Muslim Minority', 'Jain Minority']

if 'df' in locals() and not df.empty:
    predict_college(student_rank_input, selected_round_input, student_category_input, df)
else:
    print("DataFrame 'df' not found or is empty. Please ensure the data loading and cleaning steps were executed.")


Colleges eligible for student rank 300000 (adjusted to 330000) in R2 for category OPEN (including OPEN category institutes):


,Institute Name,State,Category,R2 Rank,Fee
20,"Yenepoya, Mangalore",Karnataka,OPEN,330106.0,2200000.0
19,Sri Siddhartha AcademyT Begur,Karnataka,OPEN,368629.0,1660000.0
7,MM Inst. Mullana,Haryana,OPEN,370783.0,1800000.0
16,Raja Rajeswari Bengaluru,Karnataka,OPEN,386653.0,2450000.0
60,"Malla Reddy IMS, Hyderabad",Telangana,OPEN,394117.0,2150000.0
26,Dr. DY Patil Kolhapur,Maharashtra,OPEN,397433.0,2410000.0
4,SBKS Vidyapeeth,Gujarat,OPEN,398037.0,2275000.0
59,"Malla Reddy (Women), Hyderabad",Telangana,OPEN,424521.0,2050000.0
28,"Dr. DY Patil, Pune",Maharashtra,OPEN,439258.0,2700000.0
27,"Dr. DY Patil, Navi Mumbai",Maharashtra,OPEN,457967.0,2700000.0
